# Data Ingestion Phase
This notebook describes steps required to store, ingest and validate different city datasets

## 0. Downloading Datasets

The raw data file structure is as follows for now:

- `data/raw/taxi_trips/yellow`
- `data/raw/taxi_trips/green`
- `data/raw/taxi_zones`
- `data/raw/weather`
- `data/raw/air_quality`

Note: we are only going to use data for year 2025 in week 1.

### 0a. Downloading Taxi Trips

Downloading taxi trip data parquet files (yellow and green zones):

In [8]:
from pathlib import Path
from urllib.request import urlretrieve

# Source: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
YEAR = 2025
MONTHS = range(1, 6)  # Jan–May currently published for 2026
COLORS = ("yellow", "green")

root = Path.cwd().resolve()
if not (root / "data" / "raw").exists():
    root = root.parent

raw = root / "data" / "raw" / "taxi_trips"

for color in COLORS:
    out_dir = raw / color
    out_dir.mkdir(parents=True, exist_ok=True)

    for month in MONTHS:
        name = f"{color}_tripdata_{YEAR}-{month:02d}.parquet"
        dest = out_dir / name
        if dest.exists():
            print(f"skip  {dest.relative_to(root)}")
            continue

        print(f"get   {name}")
        urlretrieve(f"{BASE_URL}/{name}", dest)
        print(f"saved {dest.relative_to(root)}")


get   yellow_tripdata_2025-01.parquet
saved data/raw/taxi_trips/yellow/yellow_tripdata_2025-01.parquet
get   yellow_tripdata_2025-02.parquet
saved data/raw/taxi_trips/yellow/yellow_tripdata_2025-02.parquet
get   yellow_tripdata_2025-03.parquet
saved data/raw/taxi_trips/yellow/yellow_tripdata_2025-03.parquet
get   yellow_tripdata_2025-04.parquet
saved data/raw/taxi_trips/yellow/yellow_tripdata_2025-04.parquet
get   yellow_tripdata_2025-05.parquet
saved data/raw/taxi_trips/yellow/yellow_tripdata_2025-05.parquet
get   green_tripdata_2025-01.parquet
saved data/raw/taxi_trips/green/green_tripdata_2025-01.parquet
get   green_tripdata_2025-02.parquet
saved data/raw/taxi_trips/green/green_tripdata_2025-02.parquet
get   green_tripdata_2025-03.parquet
saved data/raw/taxi_trips/green/green_tripdata_2025-03.parquet
get   green_tripdata_2025-04.parquet
saved data/raw/taxi_trips/green/green_tripdata_2025-04.parquet
get   green_tripdata_2025-05.parquet
saved data/raw/taxi_trips/green/green_tripdata_2

### 0b. Downloading Weather Data

Hourly observations for NYC Central Park (USAF-WBAN `72505394728`) from NOAA [ISD / Global Hourly](https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database). Prefer 2026; if that year is not published yet, fall back to the latest available year:

In [9]:
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlretrieve

# Source: https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database
# NYC Central Park (USAF-WBAN)
BASE_URL = "https://www.ncei.noaa.gov/data/global-hourly/access"
STATION_ID = "72505394728"
YEAR = 2025

root = Path.cwd().resolve()
if not (root / "data" / "raw").exists():
    root = root.parent

out_dir = root / "data" / "raw" / "weather"
out_dir.mkdir(parents=True, exist_ok=True)

# ISD global-hourly for 2026 is not published yet — fall back to the latest year that exists.
year = YEAR
while year >= YEAR - 2:
    name = f"{STATION_ID}_{year}.csv"
    dest = out_dir / name
    if dest.exists():
        print(f"skip  {dest.relative_to(root)}")
        break
    url = f"{BASE_URL}/{year}/{STATION_ID}.csv"
    try:
        print(f"get   {name}")
        urlretrieve(url, dest)
        print(f"saved {dest.relative_to(root)}")
        break
    except HTTPError as exc:
        if exc.code != 404:
            raise
        print(f"miss  {name} (HTTP 404)")
        year -= 1
else:
    raise FileNotFoundError(f"No ISD global-hourly CSV for station {STATION_ID}")


get   72505394728_2025.csv
saved data/raw/weather/72505394728_2025.csv


### 0c. Downloading Air Quality Data

Hourly PM2.5 (parameter `88101`) from EPA [AirData pre-generated files](https://aqs.epa.gov/aqsweb/airdata/download_files.html) for 2026:

In [13]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

# Source: https://aqs.epa.gov/aqsweb/airdata/download_files.html
# Hourly PM2.5 FRM/FEM (parameter code 88101)
YEAR = 2025
URL = f"https://aqs.epa.gov/aqsweb/airdata/hourly_88101_{YEAR}.zip"

root = Path.cwd().resolve()
if not (root / "data" / "raw").exists():
    root = root.parent

out_dir = root / "data" / "raw" / "air_quality"
out_dir.mkdir(parents=True, exist_ok=True)

zip_path = out_dir / f"hourly_88101_{YEAR}.zip"
csv_name = f"hourly_88101_{YEAR}.csv"
csv_path = out_dir / csv_name

if csv_path.exists():
    print(f"skip  {csv_path.relative_to(root)}")
else:
    if not zip_path.exists():
        print(f"get   {zip_path.name}")
        urlretrieve(URL, zip_path)
        print(f"saved {zip_path.relative_to(root)}")
    else:
        print(f"reuse {zip_path.relative_to(root)}")

    with ZipFile(zip_path) as zf:
        zf.extract(csv_name, out_dir)
    print(f"unzip {csv_path.relative_to(root)}")


get   hourly_88101_2025.zip
saved data/raw/air_quality/hourly_88101_2025.zip
unzip data/raw/air_quality/hourly_88101_2025.csv


### 0d. Downloading Taxi Zone Lookup Table

TLC [taxi zone lookup CSV](https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv):

In [14]:
from pathlib import Path
from urllib.request import urlretrieve

# Source: https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

root = Path.cwd().resolve()
if not (root / "data" / "raw").exists():
    root = root.parent

out_dir = root / "data" / "raw" / "taxi_zones"
out_dir.mkdir(parents=True, exist_ok=True)

dest = out_dir / "taxi_zone_lookup.csv"
if dest.exists():
    print(f"skip  {dest.relative_to(root)}")
else:
    print(f"get   {dest.name}")
    urlretrieve(URL, dest)
    print(f"saved {dest.relative_to(root)}")


get   taxi_zone_lookup.csv
saved data/raw/taxi_zones/taxi_zone_lookup.csv


## 1. Loading Datasets into Spark

### 1a. Configure Spark

Create a local Spark session with Delta Lake support.


In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

for key in ("JAVA_HOME", "SPARK_HOME", "PYSPARK_SUBMIT_ARGS"):
    os.environ.pop(key, None)

_java_candidates = (
    Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    Path("/opt/homebrew/opt/openjdk@17"),
    Path("/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
)
java_home = next((p for p in _java_candidates if (p / "bin" / "java").is_file()), None)
if java_home is None:
    raise RuntimeError("OpenJDK 17 not found. Install with: brew install openjdk@17")

os.environ["JAVA_HOME"] = str(java_home)
os.environ["PATH"] = f"{java_home / 'bin'}:" + os.environ.get("PATH", "")
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
subprocess.run([str(java_home / "bin" / "java"), "-version"], check=True, capture_output=True)

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent

site_pkgs = sorted(glob.glob(str(ROOT / ".venv" / "lib" / "python*" / "site-packages")))
if not site_pkgs:
    raise RuntimeError("Missing .venv packages. Run: .venv/bin/pip install -r requirements.txt")
site = site_pkgs[-1]
if site in sys.path:
    sys.path.remove(site)
sys.path.insert(0, site)

from pyspark import SparkContext
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip


def create_spark():
    active = SparkSession.getActiveSession()
    if active is not None:
        active.stop()
    if SparkContext._active_spark_context is not None:
        SparkContext._active_spark_context.stop()

    builder = (
        SparkSession.builder.appName("ingestion-bronze")
        .master("local[*]")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config(
            "spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog",
        )
    )
    return configure_spark_with_delta_pip(builder).getOrCreate()


spark = create_spark()
spark.sparkContext.setLogLevel("WARN")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/juozas/.ivy2/cache
The jars for the packages stored in: /Users/juozas/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3b88d144-4c20-46b7-a4a6-33990eb32109;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
:: resolution report :: resolve 121ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from local-m2-cache in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default   

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/juozas/Documents/Projects/kth/id2221_labs/id2221-labs


### 1b. Partitioning strategy

Each dataset lands as its own Delta table under `data/lake/bronze/`.

| Table | Partition columns | Why |
| --- | --- | --- |
| `taxi_trips` | `taxi_type`, `pickup_date` | Color filter + daily time pruning for trip analytics |
| `weather` | `observation_date` | Hourly series almost always queried by day/range |
| `air_quality` | `state_code`, `measurement_date` | City queries prune to NY (`36`); date helps time joins |
| `taxi_zones` | _(none)_ | Tiny lookup table — partitioning would only add overhead |

Partition columns are derived from source timestamps (or state) **before** the Delta write so Spark can prune on read without rewriting later.


### 1c. Shared helpers

Read raw files, validate that required columns exist with compatible types, then write bronze Delta tables.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional

from pyspark.sql import DataFrame, functions as F
from pyspark.sql import types as T


RAW = ROOT / "data" / "raw"
BRONZE = ROOT / "data" / "lake" / "bronze"
BRONZE.mkdir(parents=True, exist_ok=True)


COMPATIBLE = {
    "integer": (T.ByteType, T.ShortType, T.IntegerType, T.LongType),
    "long": (T.ByteType, T.ShortType, T.IntegerType, T.LongType),
    "double": (T.FloatType, T.DoubleType, T.DecimalType, T.IntegerType, T.LongType),
    "string": (T.StringType, T.IntegerType, T.LongType, T.DoubleType, T.FloatType, T.TimestampType, T.DateType, T.BooleanType),
    "timestamp": (T.TimestampType, T.TimestampNTZType, T.StringType, T.DateType, T.LongType),
    "date": (T.DateType, T.TimestampType, T.StringType),
}


@dataclass
class SchemaCheck:
    ok: bool
    missing: list
    type_mismatches: list


def validate_schema(df: DataFrame, expected: dict, dataset: str) -> SchemaCheck:
    """Require expected columns; allow extra columns; check types."""
    actual = {f.name: f.dataType for f in df.schema.fields}
    missing = [c for c in expected if c not in actual]
    mismatches = []

    for col, expected_type in expected.items():
        if col not in actual:
            continue
        allowed = COMPATIBLE.get(expected_type)
        if allowed and not isinstance(actual[col], allowed):
            mismatches.append(
                f"{col}: got {actual[col].simpleString()}, expected {expected_type}"
            )

    check = SchemaCheck(ok=not missing and not mismatches, missing=missing, type_mismatches=mismatches)
    print(f"[{dataset}] schema ok={check.ok}")
    if check.missing:
        print("  missing:", check.missing)
    if check.type_mismatches:
        print("  mismatches:", check.type_mismatches)
    if not check.ok:
        raise ValueError(f"Schema validation failed for {dataset}")
    return check


# ---- writers --------------------------------------------------------------

def delta_safe_name(name: str) -> str:
    return (
        name.strip().replace(" ", "_").replace(",", "_").replace(";", "_").replace("{", "_").replace("}", "_").replace("(", "_").replace(")", "_").replace("\n", "_").replace("\t", "_").replace("=", "_")
    )


def with_delta_safe_columns(df: DataFrame) -> DataFrame:
    out = df
    for col in df.columns:
        safe = delta_safe_name(col)
        if safe != col:
            out = out.withColumnRenamed(col, safe)
    return out


def write_bronze(df: DataFrame, table_name: str, partition_by: Optional[list[str]] = None) -> Path:
    out = BRONZE / table_name
    # normalize column names (so delta wouldn't complain) and add ingestion timestamp
    framed = with_delta_safe_columns(df).withColumn("_ingested_at", F.current_timestamp())
    writer = (
        framed.write.format("delta")
        .mode("overwrite") # overwrite existing data (we might want to reconsider this given the nature of timeseries data)
        .option("overwriteSchema", "true") # if schema changed, overwrite it with the new schema
    )
    parts = [c for c in (partition_by or []) if c in framed.columns]
    if parts:
        writer = writer.partitionBy(*parts)
    writer.save(str(out))
    return out


def show_table(table_name: str, n: int = 3) -> None:
    path = BRONZE / table_name
    df = spark.read.format("delta").load(str(path))
    print(f"{table_name}: {df.count()} rows @ {path.relative_to(ROOT)}")
    df.show(n, truncate=False)


### 1d. Ingest `taxi_zones`

Small dimension table — validate lookup columns, write unpartitioned Delta.


In [3]:
taxi_zones_expected = {
    "LocationID": "integer",
    "Borough": "string",
    "Zone": "string",
    "service_zone": "string",
}

taxi_zones_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "taxi_zones"))
)

validate_schema(taxi_zones_raw, taxi_zones_expected, "taxi_zones")
write_bronze(taxi_zones_raw, "taxi_zones", partition_by=[])
show_table("taxi_zones")


[taxi_zones] schema ok=True


26/09/06 17:50:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


taxi_zones: 265 rows @ data/lake/bronze/taxi_zones
+----------+-------+-----------------------+------------+--------------------------+
|LocationID|Borough|Zone                   |service_zone|_ingested_at              |
+----------+-------+-----------------------+------------+--------------------------+
|1         |EWR    |Newark Airport         |EWR         |2026-09-06 15:50:21.286538|
|2         |Queens |Jamaica Bay            |Boro Zone   |2026-09-06 15:50:21.286538|
|3         |Bronx  |Allerton/Pelham Gardens|Boro Zone   |2026-09-06 15:50:21.286538|
+----------+-------+-----------------------+------------+--------------------------+
only showing top 3 rows



### 1e. Ingest `weather`

NOAA ISD hourly CSV. Partition by `observation_date` (derived from `DATE`) for time-range reads.


In [4]:
weather_expected = {
    "STATION": "string",
    "DATE": "string",
    "TMP": "string",
    "WND": "string",
}

weather_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "weather"))
)

validate_schema(weather_raw, weather_expected, "weather")

weather_bronze = weather_raw.withColumn(
    "observation_date",
    F.to_date(F.col("DATE")),
)

write_bronze(weather_bronze, "weather", partition_by=["observation_date"])
show_table("weather")


[weather] schema ok=True


weather: 7722 rows @ data/lake/bronze/weather
+-----------+-------------------+------+--------+---------+---------+---------------------------+-----------+---------+---------------+--------------+-----------+------------+-------+-------+-------+-----------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+------------------+----+----+-----------------+----+----+----+-----------------------------------+----+----+----+----+----+----+----+----+----+----+----+----+---------------+----------------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+--------------------------------------------------------------------------------------------------------+----+----------------+--------------------------+
|STATION    |DATE               |SOURCE|LATITUDE|LONGITUDE|ELEVATION|NAME                       |REPORT_TYPE|CALL_SIGN|QUALITY_CONTROL|WND         

### 1f. Ingest `air_quality`

EPA hourly PM2.5. Keep New York state rows only (`State Code = 36`) for the NYC platform.

Partition by `state_code` + `measurement_date` so city/time filters prune files on both axes.


In [ ]:
air_quality_expected = {
    "State Code": "integer",
    "County Code": "integer",
    "Site Num": "integer",
    "Parameter Name": "string",
    "Date GMT": "string",
    "Time GMT": "string",
    "Sample Measurement": "double",
    "Units of Measure": "string",
}

air_quality_raw = (
    spark.read.option("header", True)
    .option("inferSchema", True)
    .csv(str(RAW / "air_quality" / "hourly_88101_2025.csv"))
)

validate_schema(air_quality_raw, air_quality_expected, "air_quality")

# select nyc only
air_quality_ny = air_quality_raw.filter(F.col("State Code") == 36)

air_quality_bronze = (
    air_quality_ny
    .withColumn("state_code", F.col("State Code"))
    .withColumn(
        "measurement_date",
        F.to_date(F.col("Date GMT"), "yyyy-MM-dd"),
    )
    # Drop originals so Delta partition columns are unambiguous after name sanitizing.
    .drop("State Code")
)

write_bronze(
    air_quality_bronze,
    "air_quality",
    partition_by=["state_code", "measurement_date"],
)
show_table("air_quality")


[air_quality] schema ok=True


air_quality: 173172 rows @ data/lake/bronze/air_quality
+-----------+--------+--------------+---+--------+---------+-----+------------------------+----------+-------------------+----------+-------------------+------------------+---------------------------+---+-----------+---------+-----------+-----------+----------------------------------------------------------------------------------+----------+-----------+-------------------+----------+----------------+--------------------------+
|County_Code|Site_Num|Parameter_Code|POC|Latitude|Longitude|Datum|Parameter_Name          |Date_Local|Time_Local         |Date_GMT  |Time_GMT           |Sample_Measurement|Units_of_Measure           |MDL|Uncertainty|Qualifier|Method_Type|Method_Code|Method_Name                                                                       |State_Name|County_Name|Date_of_Last_Change|state_code|measurement_date|_ingested_at              |
+-----------+--------+--------------+---+--------+---------+-----+--------------

### 1g. Ingest `taxi_trips`

Yellow and green TLC parquet files share most columns; pickup/dropoff names differ (`tpep_*` vs `lpep_*`).

Normalize to one schema, tag `taxi_type`, derive `pickup_date`, then partition by `taxi_type` + `pickup_date`.


In [6]:
yellow_expected = {
    "VendorID": "integer",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "long",
    "trip_distance": "double",
    "PULocationID": "long",
    "DOLocationID": "long",
    "fare_amount": "double",
    "total_amount": "double",
}

green_expected = {
    "VendorID": "integer",
    "lpep_pickup_datetime": "timestamp",
    "lpep_dropoff_datetime": "timestamp",
    "passenger_count": "long",
    "trip_distance": "double",
    "PULocationID": "long",
    "DOLocationID": "long",
    "fare_amount": "double",
    "total_amount": "double",
}


def normalize_trips(df: DataFrame, taxi_type: str) -> DataFrame:
    """Align yellow/green pickup-dropoff names and add partition columns."""
    if taxi_type == "yellow":
        out = (
            df.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
            .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
        )
    else:
        out = (
            df.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime")
            .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")
        )

    return (
        out.withColumn("taxi_type", F.lit(taxi_type))
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )


yellow_raw = spark.read.parquet(str(RAW / "taxi_trips" / "yellow"))
green_raw = spark.read.parquet(str(RAW / "taxi_trips" / "green"))

validate_schema(yellow_raw, yellow_expected, "taxi_trips/yellow")
validate_schema(green_raw, green_expected, "taxi_trips/green")

taxi_trips_bronze = normalize_trips(yellow_raw, "yellow").unionByName(
    normalize_trips(green_raw, "green"),
    allowMissingColumns=True,
)

write_bronze(
    taxi_trips_bronze,
    "taxi_trips",
    partition_by=["taxi_type", "pickup_date"],
)
show_table("taxi_trips")


[taxi_trips/yellow] schema ok=True
[taxi_trips/green] schema ok=True


taxi_trips: 20014441 rows @ data/lake/bronze/taxi_trips
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+---------+-----------+---------+---------+--------------------------+
|VendorID|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|taxi_type|pickup_date|ehail_fee|trip_type|_ingested_at              |
+--------+-------------------+-------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+-----------

### 1h. Bronze summary

Confirm all four Delta tables exist and report row counts.


In [8]:
tables = ["taxi_zones", "weather", "air_quality", "taxi_trips"]

print("Bronze Delta tables")
print("-" * 40)
for name in tables:
    path = BRONZE / name
    df = spark.read.format("delta").load(str(path))
    parts = sorted({p.name.split("=")[0] for p in path.glob("*/") if "=" in p.name})
    print(f"{name:14} rows={df.count():>12,}  partitions={parts or ['(none)']}")


Bronze Delta tables
----------------------------------------
taxi_zones     rows=         265  partitions=['(none)']
weather        rows=       7,722  partitions=['observation_date']
air_quality    rows=     173,172  partitions=['state_code']
taxi_trips     rows=  20,014,441  partitions=['taxi_type']
